# Lab 05.2 — Parameters That Can Be Optimized
The first notebook compares optimization methods. This notebook shows different neural network parameters that can be optimized using **GridSearchCV**.

Run the setup cells first, then run the examples you want to explore. Each example uses the same baseline model and changes only the listed parameters. Running every example can take time.

## 1. Imports

In [ ]:
# %pip install tensorflow scikeras scikit-learn numpy pandas
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from scikeras.wrappers import KerasClassifier


## 2. Prepare the data
Keep a separate test set for the final evaluation. In this dataset, zero measurements in columns 1–5 are treated as missing; zero pregnancies are valid.

In [ ]:
datafile = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
dataset = np.loadtxt(datafile, delimiter=',')
X = dataset[:, :8].copy()
y = dataset[:, 8].astype(int)
for column in [1, 2, 3, 4, 5]:
    X[X[:, column] == 0, column] = np.nan

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=7
)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=7)


## 3. Define the model
All examples use this function. Its arguments control the hidden layers. The binary output remains one sigmoid neuron.

We return an uncompiled model so SciKeras can select the optimizer.

In [ ]:
def create_model(neurons=16, hidden_layers=1, activation='relu',
                 init_mode='glorot_uniform', dropout_rate=0.0,
                 weight_constraint=None, l2_strength=0.0):
    model = tf.keras.Sequential()
    model.add(tf.keras.Input(shape=(8,)))

    for layer in range(hidden_layers):
        constraint = None
        if weight_constraint is not None:
            constraint = tf.keras.constraints.MaxNorm(weight_constraint)

        model.add(tf.keras.layers.Dense(
            neurons, activation=activation,
            kernel_initializer=init_mode,
            kernel_constraint=constraint,
            kernel_regularizer=tf.keras.regularizers.L2(l2_strength)
        ))
        model.add(tf.keras.layers.Dropout(dropout_rate))

    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    return model


## 4. Define the baseline
The pipeline fills missing values and standardizes features using only the training part of each CV fold.

Parameter names use these prefixes:
- `nn__epochs`: a training setting of the neural network.
- `nn__model__neurons`: an argument of `create_model`.
- `nn__optimizer__learning_rate`: a setting of the optimizer.

All examples start from this baseline; earlier best values are not automatically carried forward. The score is **balanced accuracy**, the average recall of the two classes.

In [ ]:
nn = KerasClassifier(
    model=create_model,
    optimizer=tf.keras.optimizers.Adam,
    loss='binary_crossentropy',
    epochs=30, batch_size=32,
    verbose=0, random_state=7
)
model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('nn', nn)
])


## 5. Batch size and epochs

**Batch size** controls how many samples contribute to one weight update. Smaller batches produce more updates per epoch and noisier gradient estimates; larger batches use more memory and fewer updates. Neither is always better.

**Epochs** controls how many times the training data are processed. Too few can leave the model undertrained; too many can increase overfitting and runtime. More epochs do not guarantee better validation performance.

In [ ]:
param_grid = {'nn__batch_size': [16, 32, 64], 'nn__epochs': [10, 30, 60]}

grid = GridSearchCV(
    model, param_grid, cv=cv, scoring='balanced_accuracy', n_jobs=1
)
grid.fit(X_train, y_train)

print('Best score:', grid.best_score_)
print('Best parameters:', grid.best_params_)


## 6. Optimizer
The **optimizer** determines how gradients update weights. SGD takes basic gradient steps; RMSprop adapts step sizes; Adam combines adaptive scaling with gradient moment estimates.

Here each optimizer uses its default learning rate. This compares default configurations; tuning learning rate can change which optimizer performs best.

In [ ]:
param_grid = {'nn__optimizer': [tf.keras.optimizers.SGD, tf.keras.optimizers.RMSprop, tf.keras.optimizers.Adam]}

grid = GridSearchCV(
    model, param_grid, cv=cv, scoring='balanced_accuracy', n_jobs=1
)
grid.fit(X_train, y_train)

print('Best score:', grid.best_score_)
print('Best parameters:', grid.best_params_)


## 7. Learning rate and momentum

**Learning rate** sets the size of each update. A very small value can train slowly; a very large value can overshoot good solutions or destabilize training.

**Momentum** in SGD carries part of the previous update into the next step. It can accelerate consistent movement and reduce oscillation, but a strong value can also overshoot. Its effect depends on learning rate.

This experiment deliberately uses **SGD**, because the `momentum` argument is not shared by every optimizer.

In [ ]:
param_grid = {'nn__optimizer': [tf.keras.optimizers.SGD],
              'nn__optimizer__learning_rate': [0.001, 0.01, 0.1],
              'nn__optimizer__momentum': [0.0, 0.9]}

grid = GridSearchCV(
    model, param_grid, cv=cv, scoring='balanced_accuracy', n_jobs=1
)
grid.fit(X_train, y_train)

print('Best score:', grid.best_score_)
print('Best parameters:', grid.best_params_)


## 8. Weight initialization

**Weight initialization** controls the starting weights. Glorot initialization accounts for input and output width; He initialization accounts for input width and is commonly paired with ReLU. An unsuitable weight scale can make gradients shrink or grow excessively.

Zero initialization is excluded from the normal search: identical hidden neurons can remain symmetric and fail to learn different representations. The output initializer is held fixed.

In [ ]:
param_grid = {'nn__model__init_mode': ['glorot_uniform', 'glorot_normal', 'he_uniform', 'he_normal']}

grid = GridSearchCV(
    model, param_grid, cv=cv, scoring='balanced_accuracy', n_jobs=1
)
grid.fit(X_train, y_train)

print('Best score:', grid.best_score_)
print('Best parameters:', grid.best_params_)


## 9. Hidden-layer activation

The **activation function** controls the hidden neurons' response and introduces nonlinearity. ReLU is simple but can produce inactive neurons; tanh is bounded and may saturate; sigmoid can also saturate; ELU allows negative outputs.

`linear` is included as a comparison: with no nonlinear hidden activation, the network's pre-sigmoid output remains an affine function, even if more hidden layers are added. Softmax is kept out of this hidden-layer search; the binary output remains sigmoid.

In [ ]:
param_grid = {'nn__model__activation': ['relu', 'tanh', 'sigmoid', 'elu', 'linear']}

grid = GridSearchCV(
    model, param_grid, cv=cv, scoring='balanced_accuracy', n_jobs=1
)
grid.fit(X_train, y_train)

print('Best score:', grid.best_score_)
print('Best parameters:', grid.best_params_)


## 10. Dropout rate and weight constraint

**Dropout rate** is the fraction of hidden units randomly set to zero during training. It can reduce reliance on particular units and help limit overfitting. Excessive dropout can cause underfitting. Dropout is inactive during prediction.

**MaxNorm** limits the norm of each hidden neuron's incoming weight vector after updates. A smaller limit imposes a stronger restriction; `None` disables the constraint. It does not clamp each weight individually.

In [ ]:
param_grid = {'nn__model__dropout_rate': [0.0, 0.2, 0.4],
              'nn__model__weight_constraint': [None, 2.0]}

grid = GridSearchCV(
    model, param_grid, cv=cv, scoring='balanced_accuracy', n_jobs=1
)
grid.fit(X_train, y_train)

print('Best score:', grid.best_score_)
print('Best parameters:', grid.best_params_)


## 11. Number of neurons

**Neurons** controls the width of each hidden layer. More neurons increase representational capacity and the number of trainable weights. Too few can underfit; too many may overfit and increase runtime. Larger is not automatically better.

In [ ]:
param_grid = {'nn__model__neurons': [8, 16, 32, 64]}

grid = GridSearchCV(
    model, param_grid, cv=cv, scoring='balanced_accuracy', n_jobs=1
)
grid.fit(X_train, y_train)

print('Best score:', grid.best_score_)
print('Best parameters:', grid.best_params_)


## 12. Number of hidden layers

**Hidden layers** controls network depth. Additional nonlinear layers can represent more complex relationships, but increase computation and can make optimization harder. In this example each hidden layer has the same width, controlled by `neurons`.

In [ ]:
param_grid = {'nn__model__hidden_layers': [1, 2, 3]}

grid = GridSearchCV(
    model, param_grid, cv=cv, scoring='balanced_accuracy', n_jobs=1
)
grid.fit(X_train, y_train)

print('Best score:', grid.best_score_)
print('Best parameters:', grid.best_params_)


## 13. L2 regularization

**L2 strength** adds a penalty proportional to the squared hidden weights to the training loss. Larger values encourage smaller weights and may reduce overfitting; excessive regularization can underfit. Zero disables the penalty.

Unlike MaxNorm, L2 is a soft loss penalty rather than a hard norm limit. The output layer is held fixed in this example.

In [ ]:
param_grid = {'nn__model__l2_strength': [0.0, 0.0001, 0.001, 0.01]}

grid = GridSearchCV(
    model, param_grid, cv=cv, scoring='balanced_accuracy', n_jobs=1
)
grid.fit(X_train, y_train)

print('Best score:', grid.best_score_)
print('Best parameters:', grid.best_params_)


## 14. Inspect all results
Run this after any example to see every tested combination. `std_test_score` is the variation between CV folds, not a confidence interval.

In [ ]:
results = pd.DataFrame(grid.cv_results_)
display(results[['params', 'mean_test_score', 'std_test_score']]
        .sort_values('mean_test_score', ascending=False))


## 15. Optional: optimize several parameters together
This grid has **2 × 2 × 2 = 8 combinations**, requiring 24 CV training runs and one final refit. Run this cell only if you want the additional experiment.

With many parameters, combination counts multiply quickly. Independently selected best values may not form the best combination, because parameters interact.

In [ ]:
param_grid = {
    'nn__model__neurons': [16, 32],
    'nn__model__dropout_rate': [0.0, 0.2],
    'nn__batch_size': [16, 32]
}

grid = GridSearchCV(
    model, param_grid, cv=cv, scoring='balanced_accuracy', n_jobs=1
)
grid.fit(X_train, y_train)

print('Best score:', grid.best_score_)
print('Best parameters:', grid.best_params_)


## 16. Final test
Each example overwrites `grid`. Before testing, rerun the example you selected using **CV results only**. Its best model is already refitted on the full training set.

Uncomment the lines below after completing tuning. Use the test set once; do not choose parameters using its score.

In [ ]:
# y_pred = grid.best_estimator_.predict(X_test)
# print('Test balanced accuracy:', balanced_accuracy_score(y_test, y_pred))


## Short tasks
1. Choose two examples and explain what their parameters change.
2. Compare all tested values. Is a larger value always better?
3. Explain why separate best values might not work best together.
4. Count the combinations in a grid of your own before running it.

The candidate lists are teaching examples, not universal best values. Seeds help repeatability, but results can still vary across hardware and software versions.

### Documentation
- [SciKeras parameter routing](https://adriangb.com/scikeras/stable/advanced.html)
- [Keras regularizers](https://keras.io/api/layers/regularizers/)
- [Keras constraints](https://keras.io/api/layers/constraints/)
- [Keras dropout](https://keras.io/api/layers/regularization_layers/dropout/)